In [1]:
import numpy as np
import pandas as pd

In [2]:
### Collecting the data from different sources

#### Train Data 
data1=pd.read_csv('..\data\solubility_water\BNNLab_water_solubility_data.csv')             #BNN Lab 
data2=pd.read_csv('..\data\solubility_water\Gihan_water_solubility.csv',encoding='latin1') #Gihan
data3=pd.read_csv('..\data\solubility_water\Xian_Zeng_water_solubility.csv')               #Xian Zeng
data4=pd.read_csv('..\data\solubility_water\Sorkun_water_solubility.csv')                  #Sorkun (AqSolDB)

### Test data 
test_set=pd.read_csv('..\data\solubility_water\Huuskonen_water_solubility.csv')            #Huuskonen
### Size of each dataset 
print(data1.shape)
print(data2.shape)
print(data3.shape)
print(data4.shape)
print(test_set.shape)

(900, 10)
(11862, 26)
(9955, 6)
(6154, 9)
(1291, 6)


In [3]:
### In order to merge the data all the dataset should have identical column name ... 
data1=data1[['StdInChIKey','Smiles','LogS']]
## Changing the column name to be the identical with other dataset 
data1=data1.rename(columns={ "StdInChIKey":"InChIKey" ,"Smiles":"SMILES"})

### Selecting the requiered column 
data2=data2[['Standard InChIKey','SMILES','LogS']]
## Changing the column name to be the identical with other dataset 
data2=data2.rename(columns={ "Standard InChIKey":"InChIKey"})#

### Changing the column name ...

data3=data3[['InChIKey','SMILES','logS']]
data3 = data3.rename(columns={"logS":"LogS"})

data4=data4[['InChIKey','SMILES','Solubility']]
data4 = data4.rename(columns={"Solubility":"LogS"})

test_set=test_set[['InChIKey','SMILES','Solubility']]
test_set = test_set.rename(columns={"Solubility":"LogS"})

In [ ]:
### Merging the train dataset to one dataframe 
frames = [data1,data2,data3,data4]
train_set = pd.concat(frames)
### Size of the train and test dataset before preprocess....
print('Size of the train dataset :',len(train_set))
print('Size of the test dataset :',len(test_set))


train_set.to_csv('../data/solubility_water/train_water_final.csv')
test_set.to_csv('../data/solubility_water/test_water_final.csv')

Size of the train dataset : 28871
Size of the test dataset : 1291


In [5]:
train_set.reset_index(drop=True)
test_set.reset_index(drop=True)

,InChIKey,SMILES,LogS
0,OFBQJSOFQDEBGM-UHFFFAOYSA-N,CCCCC,-3.18
1,RGSFGYAAUTVSQA-UHFFFAOYSA-N,C1CCCC1,-2.64
2,VLKZOEOYAKHREP-UHFFFAOYSA-N,CCCCCC,-3.84
3,AFABGHUZZDYHJO-UHFFFAOYSA-N,CCCC(C)C,-3.74
4,HNRMPXKDFBEGFZ-UHFFFAOYSA-N,CCC(C)(C)C,-3.55
...,...,...,...
1286,JXSJBGJIGXNWCI-UHFFFAOYSA-N,CCOC(=O)CC(SP(=S)(OC)OC)C(=O)OCC,-3.37
1287,SBPBAQFWLVIOKP-UHFFFAOYSA-N,CCOP(=S)(OCC)Oc1nc(Cl)c(Cl)cc1Cl,-5.49
1288,XEYBRNLFEZDVAW-UHFFFAOYSA-N,CCCCCC(O)C=CC1C(O)CC(=O)C1CC=CCCCC(=O)O,-2.47
1289,YVGGHNCTFXOJCH-UHFFFAOYSA-N,c(ccc(c1)Cl)(c1)C(c(ccc(c2)Cl)c2)C(Cl)(Cl)Cl,-7.15


In [6]:
from rdkit.Chem import MolFromSmiles as smi2mol
from rdkit.Chem import MolToSmiles as mol2smi
## Function to create canonical smiles 
def canon(smi):
    try:
        mol=smi2mol(smi, sanitize=True)
        smi_canon=mol2smi(mol, isomericSmiles=False, canonical=True)
        return(smi_canon)
    except:
        print("ERROR")
        return(smi)
    
#### Applying function to create the column with canonical smiles.  
train_set['smiles_canon'] = [canon(smi) for smi in train_set.SMILES]
test_set['smiles_canon'] = [canon(smi) for smi in test_set.SMILES]

ERROR
ERROR
ERROR
ERROR
ERROR
ERROR
ERROR
ERROR
ERROR
ERROR
ERROR
ERROR


[02:28:33] WARNING: not removing hydrogen atom without neighbors
[02:28:33] WARNING: not removing hydrogen atom without neighbors
[02:28:33] WARNING: not removing hydrogen atom without neighbors
[02:28:33] WARNING: not removing hydrogen atom without neighbors
[02:28:33] WARNING: not removing hydrogen atom without neighbors
[02:28:33] WARNING: not removing hydrogen atom without neighbors
[02:28:33] WARNING: not removing hydrogen atom without neighbors
[02:28:33] Explicit valence for atom # 5 N, 4, is greater than permitted
[02:28:33] WARNING: not removing hydrogen atom without neighbors
[02:28:33] Explicit valence for atom # 5 N, 4, is greater than permitted
[02:28:33] WARNING: not removing hydrogen atom without neighbors


ERROR
ERROR


[02:28:33] WARNING: not removing hydrogen atom without neighbors
[02:28:33] WARNING: not removing hydrogen atom without neighbors
[02:28:33] WARNING: not removing hydrogen atom without neighbors


In [7]:
test_set[0:200]

,InChIKey,SMILES,LogS,smiles_canon
0,OFBQJSOFQDEBGM-UHFFFAOYSA-N,CCCCC,-3.18,CCCCC
1,RGSFGYAAUTVSQA-UHFFFAOYSA-N,C1CCCC1,-2.64,C1CCCC1
2,VLKZOEOYAKHREP-UHFFFAOYSA-N,CCCCCC,-3.84,CCCCCC
3,AFABGHUZZDYHJO-UHFFFAOYSA-N,CCCC(C)C,-3.74,CCCC(C)C
4,HNRMPXKDFBEGFZ-UHFFFAOYSA-N,CCC(C)(C)C,-3.55,CCC(C)(C)C
...,...,...,...,...
195,KENZYIHFBRWMOD-UHFFFAOYSA-N,c1cc(Cl)c(Cl)cc1c2c(Cl)ccc(Cl)c2,-7.25,Clc1ccc(Cl)c(-c2ccc(Cl)c(Cl)c2)c1
196,QORAVNMWUNPXAO-UHFFFAOYSA-N,c1cc(Cl)cc(Cl)c1c2c(Cl)cc(Cl)cc2,-6.51,Clc1ccc(-c2ccc(Cl)cc2Cl)c(Cl)c1
197,WIDHRBRBACOVOY-UHFFFAOYSA-N,c1cc(Cl)c(Cl)cc1c2c(Cl)c(Cl)c(Cl)cc2,-7.05,Clc1ccc(-c2ccc(Cl)c(Cl)c2Cl)cc1Cl
198,MTCPZNVSDFCBBE-UHFFFAOYSA-N,Clc1cccc(Cl)c1c2c(Cl)cc(Cl)cc2Cl,-7.32,Clc1cc(Cl)c(-c2c(Cl)cccc2Cl)c(Cl)c1


In [8]:
### Calculate  the occurence of  smiles in both train and test dataset....
train_set['occurence'] = train_set.groupby('smiles_canon')['smiles_canon'].transform('count')
test_set['occurence'] = test_set.groupby('smiles_canon')['smiles_canon'].transform('count')

In [9]:
### Taking out Unique smiles from the train and test dataset 
train_set1= train_set[train_set['occurence']==1]
print(train_set1.shape)
test_set1= test_set[test_set['occurence']==1]
print(test_set1.shape)

(13432, 5)
(1273, 5)


In [10]:
### Selecting specific column of the dataframe ..
train_set1=train_set1[['InChIKey','smiles_canon','LogS','occurence']]
test_set1=test_set1[['InChIKey','smiles_canon','LogS','occurence']]

In [11]:
#### Taking out the dataframe which has duplicate smiles means more than one time occurence in the dataset ...
train_set2= train_set[train_set['occurence']>1]
print(train_set2.shape)
test_set2= test_set[test_set['occurence']>1]
print(test_set2.shape)

(15427, 5)
(18, 5)


In [12]:
#Extract duplicate rows
id1 = train_set2["smiles_canon"]
train_set2=train_set2[id1.isin(id1[id1.duplicated()])].sort_values("smiles_canon")
id2 = test_set2["smiles_canon"]
test_set2=test_set2[id2.isin(id2[id2.duplicated()])].sort_values("smiles_canon")
print(train_set2.shape)
print(test_set2.shape)

(15427, 5)
(18, 5)


In [13]:
train_set2.reset_index(drop=True)
test_set2.reset_index(drop=True)

,InChIKey,SMILES,LogS,smiles_canon,occurence
0,ITRJWOMZKQRYTA-UHFFFAOYSA-N,CC(=O)OCC(=O)C3(O)CCC4C2CCC1=CC(=O)CCC1(C)C2C(...,-4.30,CC(=O)OCC(=O)C1(O)CCC2C3CCC4=CC(=O)CCC4(C)C3C(...,2
1,ITRJWOMZKQRYTA-UHFFFAOYSA-N,CC(=O)OCC(=O)C3(O)CCC4C2CCC1=CC(=O)CCC1(C)C2C(...,-4.00,CC(=O)OCC(=O)C1(O)CCC2C3CCC4=CC(=O)CCC4(C)C3C(...,2
2,RUDATBOHQWOJDD-UHFFFAOYSA-N,CC(CCC(O)=O)C3CCC4C2C(O)CC1CC(O)CCC1(C)C2CCC34C,-3.64,CC(CCC(=O)O)C1CCC2C3C(O)CC4CC(O)CCC4(C)C3CCC12C,2
3,RUDATBOHQWOJDD-UHFFFAOYSA-N,C1C(O)CC2CC(O)C3C4CCC(C(C)CCC(=O)O)C4(C)CCC3C2...,-3.82,CC(CCC(=O)O)C1CCC2C3C(O)CC4CC(O)CCC4(C)C3CCC12C,2
4,UREBDLICKHMUKA-UHFFFAOYSA-N,C1(=O)C=C2CCC3C4CC(C)C(O)(C(=O)CO)C4(C)CC(O)C3...,-3.64,CC1CC2C3CCC4=CC(=O)C=CC4(C)C3(F)C(O)CC2(C)C1(O...,2
5,UREBDLICKHMUKA-UHFFFAOYSA-N,C1=CC(=O)C=C2CCC3C4CC(C)C(O)(C(=O)CO)C4(C)CC(O...,-3.77,CC1CC2C3CCC4=CC(=O)C=CC4(C)C3(F)C(O)CC2(C)C1(O...,2
6,KVZJLSYJROEPSQ-UHFFFAOYSA-N,C1C(C)C(C)CCC1,-4.27,CC1CCCCC1C,2
7,KVZJLSYJROEPSQ-UHFFFAOYSA-N,C1C(C)C(C)CCC1,-4.33,CC1CCCCC1C,2
8,RFZHJHSNHYIRNE-UHFFFAOYSA-N,CC(C)C(C)(O)CC,-0.85,CCC(C)(O)C(C)C,2
9,RFZHJHSNHYIRNE-UHFFFAOYSA-N,CC(C)C(O)(C)CC,-1.22,CCC(C)(O)C(C)C,2


In [14]:
train_set2=train_set2[['InChIKey','smiles_canon','LogS','occurence']]
test_set2=test_set2[['InChIKey','smiles_canon','LogS','occurence']]
train_set2.reset_index(drop=True)
test_set2.reset_index(drop=True)

,InChIKey,smiles_canon,LogS,occurence
0,ITRJWOMZKQRYTA-UHFFFAOYSA-N,CC(=O)OCC(=O)C1(O)CCC2C3CCC4=CC(=O)CCC4(C)C3C(...,-4.30,2
1,ITRJWOMZKQRYTA-UHFFFAOYSA-N,CC(=O)OCC(=O)C1(O)CCC2C3CCC4=CC(=O)CCC4(C)C3C(...,-4.00,2
2,RUDATBOHQWOJDD-UHFFFAOYSA-N,CC(CCC(=O)O)C1CCC2C3C(O)CC4CC(O)CCC4(C)C3CCC12C,-3.64,2
3,RUDATBOHQWOJDD-UHFFFAOYSA-N,CC(CCC(=O)O)C1CCC2C3C(O)CC4CC(O)CCC4(C)C3CCC12C,-3.82,2
4,UREBDLICKHMUKA-UHFFFAOYSA-N,CC1CC2C3CCC4=CC(=O)C=CC4(C)C3(F)C(O)CC2(C)C1(O...,-3.64,2
5,UREBDLICKHMUKA-UHFFFAOYSA-N,CC1CC2C3CCC4=CC(=O)C=CC4(C)C3(F)C(O)CC2(C)C1(O...,-3.77,2
6,KVZJLSYJROEPSQ-UHFFFAOYSA-N,CC1CCCCC1C,-4.27,2
7,KVZJLSYJROEPSQ-UHFFFAOYSA-N,CC1CCCCC1C,-4.33,2
8,RFZHJHSNHYIRNE-UHFFFAOYSA-N,CCC(C)(O)C(C)C,-0.85,2
9,RFZHJHSNHYIRNE-UHFFFAOYSA-N,CCC(C)(O)C(C)C,-1.22,2


In [15]:
print(train_set2.shape)
print(test_set2.shape)

(15427, 4)
(18, 4)


In [16]:
# Group by SMILES
grouped_train = train_set2.groupby('smiles_canon')
grouped_test = test_set2.groupby('smiles_canon')
# Filter groups where the maximum difference in solubility is less than or equal to 0.50
filtered_groups_train = grouped_train.filter(lambda x: x['LogS'].max() - x['LogS'].min() <= 0.50)
filtered_groups_test = grouped_test.filter(lambda x: x['LogS'].max() - x['LogS'].min() <= 0.50)
print(len(filtered_groups_train))
print(len(filtered_groups_test))

14044
18


In [17]:
### In order to eavluate to threshold followed by the dataframe we can cross check the value of difference
grouped = filtered_groups_train.groupby('smiles_canon')['LogS'].apply(lambda x: x.sort_values().tolist())

# Calculate differences in solubility within each group
diffs = grouped.apply(lambda x: [j - i for i, j in zip(x[:-1], x[1:])])

# Create a new DataFrame with SMILES, Solubility values, and differences
result = grouped.reset_index()
result.columns = ['smiles_canon', 'LogS_Values']

# Add the differences as a new column
result['Differences'] = diffs.values

result

,smiles_canon,LogS_Values,Differences
0,Br.CN(C)CCC=C1c2ccccc2Sc2ccc(F)cc21,"[-5.603798043, -5.603798043]",[0.0]
1,Br.Oc1ccc2c(c1)C13CCCCC1C(C2)N(CCc1ccccc1)CC3,"[-4.833904011, -4.833904011]",[0.0]
2,BrC(Br)(Br)Br,"[-3.157037902, -3.1400001, -3.14]","[0.017037801999999935, 9.999999983634211e-08]"
3,BrC(Br)Br,"[-1.911295104, -1.91, -1.91]","[0.0012951039999999914, 0.0]"
4,BrC(Br)C(Br)Br,"[-2.72, -2.710626523]",[0.009373477000000019]
...,...,...,...
5683,c1cnoc1,"[0.3827, 0.383489897]",[0.0007898970000000394]
5684,c1coc(-c2nc3ccccc3[nH]2)c1,"[-3.414031277, -3.414]",[3.127699999971867e-05]
5685,c1nc[nH]n1,"[0.788504777, 1.005834993]",[0.2173302159999999]
5686,c1ncc2[nH]cnc2n1,"[0.619390837, 0.62]",[0.0006091630000000237]


In [18]:
a = filtered_groups_train[['InChIKey','smiles_canon','LogS','occurence']].groupby('smiles_canon').agg({'InChIKey':lambda x: x.iloc[0],'LogS':'mean','occurence':'mean'})
a

,InChIKey,LogS,occurence
smiles_canon,,,
Br.CN(C)CCC=C1c2ccccc2Sc2ccc(F)cc21,OMHUKXIIJPBUIP-FJUODKGNSA-N,-5.603798,2.0
Br.Oc1ccc2c(c1)C13CCCCC1C(C2)N(CCc1ccccc1)CC3,VKBPZWOOYLNYSW-YGICXTQQSA-N,-4.833904,2.0
BrC(Br)(Br)Br,HJUGFYREWKUQJT-UHFFFAOYSA-N,-3.145679,3.0
BrC(Br)Br,DIKBFYAXUHHXCS-UHFFFAOYSA-N,-1.910432,3.0
BrC(Br)C(Br)Br,QXSZNDIIPUOQMB-UHFFFAOYSA-N,-2.715313,2.0
...,...,...,...
c1cnoc1,CTAPFRYPJLPFDF-UHFFFAOYSA-N,0.383095,2.0
c1coc(-c2nc3ccccc3[nH]2)c1,UYJUZNLFJAWNEZ-UHFFFAOYSA-N,-3.414016,2.0
c1nc[nH]n1,NSPMIYGKQJPBQR-UHFFFAOYSA-N,0.897170,2.0


In [19]:
# Group again by SMILES and calculate the mean of solubility
mean_solubility_df_train = filtered_groups_train.groupby('smiles_canon').agg({'InChIKey':lambda x: x.iloc[0],'LogS':'mean','occurence':'mean'}).reset_index()
mean_solubility_df_test = filtered_groups_test.groupby('smiles_canon').agg({'InChIKey':lambda x: x.iloc[0],'LogS':'mean','occurence':'mean'}).reset_index()

### Combining the dataframe with single entry and matching smiles and average the solubility values 
train_set=pd.concat([train_set1, mean_solubility_df_train], axis=0, ignore_index=True)
test_set=pd.concat([test_set1, mean_solubility_df_test], axis=0, ignore_index=True)

print(train_set.shape)
print(test_set.shape)

(19120, 4)
(1282, 4)


In [20]:
matching_smiles = pd.merge(train_set[['smiles_canon']], test_set[['smiles_canon']], on='smiles_canon')['smiles_canon']

# Remove the matching SMILES from both DataFrames
df1_filtered = train_set[~train_set['smiles_canon'].isin(matching_smiles)]

matching_smiles

0                          Sc1ncnc2nccnc12
1                             Oc1ccnc(S)n1
2                          Cc1cc(O)nc(S)n1
3        O=c1[nH]cnc2c1ncn2C1OC(CO)C(O)C1O
4       C1CCC([Sn](C2CCCCC2)C2CCCCC2)CC1.O
                       ...                
1178                              c1ccnnc1
1179                               c1ccoc1
1180                               c1ccsc1
1181                        c1cnc2ncncc2n1
1182                              c1cncnc1
Name: smiles_canon, Length: 1183, dtype: object

In [21]:
df1_filtered

,InChIKey,smiles_canon,LogS,occurence
0,BOCJQSFSGAZAPQ-UHFFFAOYSA-N,O=C1c2ccccc2C(=O)c2c(Cl)cccc21,-5.540000,1.0
1,BRBKOPJOKNSWSG-UHFFFAOYSA-N,NC(N)=NS(=O)(=O)c1ccc(N)cc1,-1.984970,1.0
2,BRPOVUNVTUCYNN-UHFFFAOYSA-N,CCC1C(N)CN1c1cc2c(cc1F)c(=O)c(C(=O)O)cn2C1CC1,-3.912000,1.0
3,BWXSXZAIQQSFPW-UHFFFAOYSA-N,CCC1(CC)OC(=O)c2cc([N+](=O)[O-])ccc21,-3.656000,1.0
4,CAQHPYSDQFDJAL-UHFFFAOYSA-N,O=C1C=Cc2ccccc2C1=NO,-2.937000,1.0
...,...,...,...,...
19115,CTAPFRYPJLPFDF-UHFFFAOYSA-N,c1cnoc1,0.383095,2.0
19116,UYJUZNLFJAWNEZ-UHFFFAOYSA-N,c1coc(-c2nc3ccccc3[nH]2)c1,-3.414016,2.0
19117,NSPMIYGKQJPBQR-UHFFFAOYSA-N,c1nc[nH]n1,0.897170,2.0
19118,KDCGOANMDULRCW-UHFFFAOYSA-N,c1ncc2[nH]cnc2n1,0.619695,2.0


In [22]:
df_concat = pd.concat([train_set, test_set])

# Drop duplicates SMILES
df_unique = df_concat.drop_duplicates(subset='smiles_canon', keep=False)

# Split the unique rows back into two DataFrames
df1_filtered = df_unique[df_unique['smiles_canon'].isin(train_set['smiles_canon'])].reset_index(drop=True)

df1_filtered

,InChIKey,smiles_canon,LogS,occurence
0,BOCJQSFSGAZAPQ-UHFFFAOYSA-N,O=C1c2ccccc2C(=O)c2c(Cl)cccc21,-5.540000,1.0
1,BRBKOPJOKNSWSG-UHFFFAOYSA-N,NC(N)=NS(=O)(=O)c1ccc(N)cc1,-1.984970,1.0
2,BRPOVUNVTUCYNN-UHFFFAOYSA-N,CCC1C(N)CN1c1cc2c(cc1F)c(=O)c(C(=O)O)cn2C1CC1,-3.912000,1.0
3,BWXSXZAIQQSFPW-UHFFFAOYSA-N,CCC1(CC)OC(=O)c2cc([N+](=O)[O-])ccc21,-3.656000,1.0
4,CAQHPYSDQFDJAL-UHFFFAOYSA-N,O=C1C=Cc2ccccc2C1=NO,-2.937000,1.0
...,...,...,...,...
17932,CTAPFRYPJLPFDF-UHFFFAOYSA-N,c1cnoc1,0.383095,2.0
17933,UYJUZNLFJAWNEZ-UHFFFAOYSA-N,c1coc(-c2nc3ccccc3[nH]2)c1,-3.414016,2.0
17934,NSPMIYGKQJPBQR-UHFFFAOYSA-N,c1nc[nH]n1,0.897170,2.0
17935,KDCGOANMDULRCW-UHFFFAOYSA-N,c1ncc2[nH]cnc2n1,0.619695,2.0


In [23]:
### Remove the smiles from train dataset with matching test dataset...
smiles_train_canon=train_set.smiles_canon
smiles_test_canon=test_set.smiles_canon

overlap1 = 0
for x in smiles_train_canon:
    if x in smiles_test_canon:
        overlap1+=1
print("%i of the train molecules are in the test set"%(overlap1))

overlap2 = 0
for x in smiles_test_canon:
    if x in smiles_train_canon:
        overlap2+=1
print("%i of the test molecules are in the train set"%(overlap2))

0 of the train molecules are in the test set
0 of the test molecules are in the train set


In [24]:
Match_rows = pd.merge(test_set, train_set, on=['smiles_canon'], how='inner')
#mergedStuff.head()
print(len(Match_rows))

1183


In [25]:
### Finding the matching smiles in the train dataset..
cond = train_set['smiles_canon'].isin(Match_rows['smiles_canon'])

#### Dropping the smiles which is same and making dataframe with uniques smiles and no smiles same in ttrain  dataset 
train_set.drop(train_set[cond].index, inplace = True)

print(train_set.shape)
print(test_set.shape)

(17937, 4)
(1282, 4)


In [26]:
### Cross check in order to find that any matching smiles exist in the train dataset with test dataset
Match_rows = pd.merge(test_set, train_set, on=['smiles_canon'], how='inner')
#mergedStuff.head()
print(len(Match_rows))

0


In [27]:
train_set.reset_index(drop=True)

,InChIKey,smiles_canon,LogS,occurence
0,BOCJQSFSGAZAPQ-UHFFFAOYSA-N,O=C1c2ccccc2C(=O)c2c(Cl)cccc21,-5.540000,1.0
1,BRBKOPJOKNSWSG-UHFFFAOYSA-N,NC(N)=NS(=O)(=O)c1ccc(N)cc1,-1.984970,1.0
2,BRPOVUNVTUCYNN-UHFFFAOYSA-N,CCC1C(N)CN1c1cc2c(cc1F)c(=O)c(C(=O)O)cn2C1CC1,-3.912000,1.0
3,BWXSXZAIQQSFPW-UHFFFAOYSA-N,CCC1(CC)OC(=O)c2cc([N+](=O)[O-])ccc21,-3.656000,1.0
4,CAQHPYSDQFDJAL-UHFFFAOYSA-N,O=C1C=Cc2ccccc2C1=NO,-2.937000,1.0
...,...,...,...,...
17932,CTAPFRYPJLPFDF-UHFFFAOYSA-N,c1cnoc1,0.383095,2.0
17933,UYJUZNLFJAWNEZ-UHFFFAOYSA-N,c1coc(-c2nc3ccccc3[nH]2)c1,-3.414016,2.0
17934,NSPMIYGKQJPBQR-UHFFFAOYSA-N,c1nc[nH]n1,0.897170,2.0
17935,KDCGOANMDULRCW-UHFFFAOYSA-N,c1ncc2[nH]cnc2n1,0.619695,2.0


In [28]:
print(train_set.shape)
print(test_set.shape)

(17937, 4)
(1282, 4)


In [29]:
train_smiles = set(train_set['smiles_canon'])
test_smiles = set(test_set['smiles_canon'])
overlap = train_smiles.intersection(test_smiles)

if overlap:
    print("There are overlapping smiles between the training and test datasets:")
    print(overlap)
else:
    print("No overlapping smiles between the training and test datasets.")

No overlapping smiles between the training and test datasets.


In [30]:
train_set = train_set.reset_index(drop=True)
test_set = test_set.reset_index(drop=True)
train_set

,InChIKey,smiles_canon,LogS,occurence
0,BOCJQSFSGAZAPQ-UHFFFAOYSA-N,O=C1c2ccccc2C(=O)c2c(Cl)cccc21,-5.540000,1.0
1,BRBKOPJOKNSWSG-UHFFFAOYSA-N,NC(N)=NS(=O)(=O)c1ccc(N)cc1,-1.984970,1.0
2,BRPOVUNVTUCYNN-UHFFFAOYSA-N,CCC1C(N)CN1c1cc2c(cc1F)c(=O)c(C(=O)O)cn2C1CC1,-3.912000,1.0
3,BWXSXZAIQQSFPW-UHFFFAOYSA-N,CCC1(CC)OC(=O)c2cc([N+](=O)[O-])ccc21,-3.656000,1.0
4,CAQHPYSDQFDJAL-UHFFFAOYSA-N,O=C1C=Cc2ccccc2C1=NO,-2.937000,1.0
...,...,...,...,...
17932,CTAPFRYPJLPFDF-UHFFFAOYSA-N,c1cnoc1,0.383095,2.0
17933,UYJUZNLFJAWNEZ-UHFFFAOYSA-N,c1coc(-c2nc3ccccc3[nH]2)c1,-3.414016,2.0
17934,NSPMIYGKQJPBQR-UHFFFAOYSA-N,c1nc[nH]n1,0.897170,2.0
17935,KDCGOANMDULRCW-UHFFFAOYSA-N,c1ncc2[nH]cnc2n1,0.619695,2.0


In [31]:
#import pubchempy as pcp
#solute_names = []
#for smiles in test_set['smiles_canon'].tolist():
#    compound = pcp.get_compounds(smiles,'smiles')
#    name = compound[0].iupac_name
#    solute_names.append(name)
#test_set['solute_name'] = solute_names
#test_set


In [32]:
#solute_names = []
#for smiles in train_set['smiles_canon'].tolist():
#    compound = pcp.get_compounds(smiles,'smiles')
#    name = compound[0].iupac_name
#    solute_names.append(name)
#train_set['solute_name'] = solute_names
#train_set


In [33]:
### Saving the data to the disk 
train_set.to_csv('../data/solubility_water/unique_train_water_final.csv', index=False)
test_set.to_csv('../data/solubility_water/unique_test_water_final.csv', index=False)

: 